In [ ]:
# 01. 도구 정의
##############################################################

# 먼저 각 에이전트가 사용할 도구를 정의함. 이 에이전트의 역할을 명확히 분리하기 위해 리서치는 검색 기능만 수행하고, 차트 생성기는 파이썬 코드 실행만 수행하도록 섦정.
# 이를 위해 웹 검색을 담당하는 TavilySearch 도구와 로컬 파이썬 코드를 실행 할 수 있는 PythonREPL 도구를 각각 준비함.
# 이렇게 도구를 분리하면 에이전트가 허용된 작업 범위를 벗어나지 않도록 제한할 수 있으며, 멀티 에이전트 협업 구조를 좀더 안정적으로 구성할 수 있음.

# 챠트 생성시에는 파이썬 코드를 직접 실행 할 수있도록 PythonREFL 을 랭체인 도구 형태로 감싼 함수를 정의함. 
# 이 함수는 문자열 형태의 파이썬 코드를 입력으로 받아, 실행 결과를 문자열로 반환함.
# 에이전트가 생성한 코드가 실제로 실행되기때문에 차트 이미지 생성이나 데이터 처리와 같은 작업을 수행할 수 있음.
# 또한 실행중 오류가 발생하더라도 예외 메시지를 반환하도록 구성해 디버깅이 가능하도록 함. 
# 최종적으로 실행이 이뤄지면 FINAL ANSWER 를 포함한 메시지를 반환한 이후 그래프 종료 판단에 활용함.

from typing import Annotated
from langchain_tavily import TavilySearch
from langchain_core.tools import tool
from langchain_experimental.utilities import PythonREPL
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

llm = init_chat_model("openai:gpt-4.1")

# 1) 검색 도구(Tavily API 필요)
tavily_tool = TavilySearch(max_results = 5)

# 2) 로컬 파이썬 실행도구(차트 생성 전용)
repl = PythonREPL()

@tool
def python_repl_tool(
    code: Annotated[str, "차트 생성을 위한 파이썬 코드"]
):
    """파이썬 코드를 실행합니다. 출력은 print(...)로 표시해야 합니다."""
    try:
        result = repl.run(code)
    except BaseException as e:
        return f"실행 실패: {repr}"

    return f"성공적으로 실행했습니다. : \n python \n {code} \n\n Stdout: {result} \n\nFINAL ANSWER"

In [ ]:
# 02. 시스템 프롬프트 & 루프 종료 판단.
##############################################################

# 멀티 에이전트 환경에서는 각 에이전트가 작업을 언제까지 수행해야 하는지 명확히 정의하지 않으면 불필요한 반복이나 루틴 루프가 발생할 수 잇음.
# 이를 방지하기 위해 모든 에이전트에 공통으로 적용할 시스템 프롬프트와 작업 종료 시점을 판단하는 로직을 정의함.

# make_system_prompt() 는 각 에이전트에게 주어진 도구만 활용하고, 작업이 끝나면 반드시 "FINAL ANSWER"를 붙이도록 지시함.
# get_next_done()는 마지막 메시지에 "FINAL ANSWER"가 포함되면 그래프 실행을 종료(END)하고, 그렇지 않으면 지정한 다음 노드를 진행시킴.

# 3) 시스템 프롬프트 템플릿
def make_system_prompt(suffix: str):
    return {
        "당신은 다른 어시스턴트와 협업하는 유능한 AI 입니다."
        "주어진 도구만 사용해 작업을 완수하세요"
        "최종 답변을 얻으면 반드시 'FINAL ANSWER'를 붙여 종료하세요"
        f"\n {suffix}"
    }

def get_next_node(last_message: BaseMessage, goto: str):
    if "FINAL ANSWER" in last_message.content:
        return END

    return goto

In [ ]:
# 03. 노드 생성 - 리서치와 차트 생성기.
##############################################################

# 이제 실제로 작업을 수행할 두개의 에이전트를 생성함.
# 이 예제에서는 역할을 명확히 분리하기 위해 각 에이전트가 사용할 수 있는 도구와 수행할 수 있는 작업을 엄격하게 제한함.
# 리서치 에이전트는 Tavily 검색 도구만 사용할수 있도록 설정돼 있으며, 웹을 통해 환율과 관련된 정보를 조사하는 역할만 담당함.
# 반면 차트 생성기 에이전트는 파이썬 실행 도구만 사용할 수있으며, 리서처가 수집한 정보를 바탕으로 차트를 생성하는 작업에만 집중함.
# 두 에이전트 모두 ReAct 패턴을 기반으로 생성되며, 시스템 프롬프트를 통해 자신의 역할과 협업 방식에 대한 지침을 전달 받음

from typing import Literal
from langchain.agents import create_agent
from langgraph.graph import MessagesState # 대화형 AI 애플리케이션(챗봇, AI 에이전트 등)을 쉽게 개발할수 있도록 제공하는 내장(Pre-built)상태 관리 클래스
from langgraph.types import Command # 상태(State)를 어떻게 업데이트할지 와 다음에 어떤 노드를 이동할지(Routing)를 하나의 객체로 결합하여 제어하는 핵심 흐름 제어 도구

# Command 객체
#   . update(dict): 그래프의 상태를 갱신할 데이터 딕셔너리
#   . goto(str|List|Send): 다음에 실행할 노드의 이름이나 경로를 지정함
#   . graph(Command.PARENT) : 멀티 에이전트 환경에서 서브그래프(Subgraph)가 부모 그래프의 특정 노드로 직접 제어권을 넘겨야 할때 사용
#   . resume(Any) : Human-in-the-loop (인간 개입 심사) 구조에서 interrupt() 에 의해 일시정지된 그래프를 다시 재개할 때 피드백 값을 전달하는 용도 

# 4) 리서치 노드 (검색 전용)
research_agent = create_agent(
    model=llm,
    tools=[tavily_tool,],
    system_prompt = make_system_prompt("당신은 리서치만 할수 있습니다. 동료 차트 생성자와 협업하세요.")
)

def research_node(state: MessagesState) -> Command[Literal["chart_generator", END]]:
    result = research_agent.invoke(state)
    goto = get_next_node(result["messages"][-1], "chart_generator")

    # 일부 모델 제약 회피: 마지막 메시지를 Human 역할로 매핑
    result["messages"][-1] = HumanMessage(content=result["messages"][-1].content, name="researcher")

    return Command(update={"messages": result["messages"]}, goto=goto)

# 차트 생성기 역시 동일한 구조로 정의되었지만 사용할 수있는 도구는 파이썬 실행 도구로 제한됨.
# chart_node 는 차트 생성기 에이전트를 실행하고, 차트 생성이 완료되어 FINAL ANSWER 가 반환되면 그래프를 종료함. 
# 그렇지 않은 경우에는 다시 리서치 노드로 제어를 넘겨 추가조사를 진행하도록 함.

# 5) 차트 생성 노드(파이썬 실행 전용)
chart_agent = create_agent(
    model = llm,
    tools = [python_repl_tool],
    system_prompt = make_system_prompt("당신은 차트 생성만 할 수 있습니다. 동료 리서치와 협업하세요.")
)

def chart_node(state: MessagesState) -> Command[Literal["research_node", END]]:
    result = chart_agent.invoke(state)

    goto = get_next_node(result["messages"][-1], "researcher")

    result["messages"][-1] = HumanMessage(content= result["messages"][-1].content, name="chart_generator")

    return Command(update={"messages": result["messages"]}, goto = goto)

In [ ]:
# 04. 워크 플로우 구성.
##############################################################

# 모든 노드와 에이전트를 정의했다면 이제 이들을 하나의 실행 흐름으로 연결함. 시작점은 리서치이며, 이후 필요 시 차트 생성기와 다시 리서처를 오가는 구조임.

from IPython.display import Image, display

# 6) 그래프 구성.
workflow = StateGraph(MessagesState)
workflow.add_node("researcher", research_node)
workflow.add_node("chart_generator", chart_node)

workflow.add_edge(START, "researcher")
graph = workflow.compile()

# 다이어 그램 표시
try :
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

# 여기서 StateGraph 라는 상태 기반의 실행 그래프를 정의하는 객체이며, 상태 타입은 MessagesState 로 통일 했음
# 이 예제에서는 그래프에 START -> researcher 엣지만 명시적으로 추가했지만, 실행 흐름은 리서처와 차트 생성기를 번갈아 오가며 정상적으로 동작함.

# graph.get_graph().draw_mermaid_png() 를 사용해 전체 워크플로우 구조를 다이어 그램으로 시각화 하면 앞에서 정의한 실행 흐름에 따라 chart_generator 노드 역시 정상적으로 연결돼 있음을 확인

In [ ]:
# 05. 실행
##############################################################

# 마지막으로 사용자의 요청을 넣어 멀티 에이전트 협업이 실제로 어떻게 진행되는지 확인함.

# 7) 실행
events = graph.stream(
    {
        "messages": [(
            "user",
            "지난 1년간 달러 환율 데이터를 조사하고, 그 데이터를 기반으로 라인 차트를 만드세요"
            "차트를 ㅅ애성했으면 작업을 종료하세요"
        )]
    },
    {      재귀
        "recursion_limit": 150 # 안전장치: 최대 스탭 제한
    }
)

# graph.stream()
# LangGraph에서 워크플로우(그래프)를 실행할 때, 전체 프로세스가 완료될 때까지 기다리지 않고 단계별 실행 상태나 LLM의 토큰 출력을 실시간으로 받아오는(Streaming) 핵심 메서드임
# ---------------------------------------------------------
#   for chunk in graph.stream(
#       input = {"messages": [("user", "안녕?")]},          # 초기 입력값
#       config = {"configurable": {"thread_id": "1"}},      # 스레드/체크포인터 설정
#       stream_mode = "updates"                             # 스트리밍 모드 선택
#   ):
#       print(chunk)
# ---------------------------------------------------------

# .input : 그래프 실행을 시작할 때 전달하는 초기 상태값
# .config : thread_id 와 같은 체크포인트 관리용 메타데이터를 전달함
# .stream_mode : 가장 중요한 옵션으로 "무엇을" 실시간으로 받아올지 결정함.

# stream_mode 종류
#   . "updates" (기본값) | {"노드이름":{"상태키":"값"}} | 각 단계에서 '변경된 상태'만 출력.
#   . "values"  | {"상태키": "값"} | 각 단계가 끝난 후의 "전체 상태 스냅샷"을 출력
#   . "messages" | (MessageChunk, Metadata) | LLM 이 답변을 생성할 때 글자(토큰) 단위로 실시간 출력함. 챗봇의 "타이핑 효과"를 구현할 때 필수적임
#   . "custom"  | 사용자가 정의한 데이터 구조 | 노드 내부에서 get_stream_writer 를 통해 개발자가 직접 발행한 커스텀 이벤트나 데이터를 스트리밍함.
#   . "debug"  | 디버그 트레이스 전체 | 노드 진입/진출, 에러 로그 등 내부 실행의 모든 상세 정보를 출력함. 개발 단계에서 흐름을 추적할 때 씀

for step in events:
    print(step) # 각 단계와 중간 결과/메시지 스트림을 확인
    print("---")

# 리서치가 환율 데이터를 수집한 뒤, 차트 생성기가 해당 데이터를 기반으로 matplotlib 코드를 실행해 차트를 생성함.
# 마지막 출력에 "FINAL ANSWER"가 포함되면 루프가 종료됨